In [1]:
import os

import numpy as np

import pandas as pd
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)

import re

In [2]:
# Read in data
matching_df = pd.read_csv('namibia_2019_matching_data_2020-10-31.csv', header=0, dtype={'Food item code in Household Survey (item_cod)': str})

In [3]:
# Strip whitespace from column names
matching_df.rename(columns=lambda x: x.strip().replace(' ', '_').replace('(', '').replace(')', ''), inplace=True)

In [4]:
# Get str cols
str_cols = matching_df.columns[0:4].to_list()

# Convert to str type
matching_df[str_cols] = matching_df[str_cols].astype(str)

# Strip whitespace in str values
matching_df[str_cols] = matching_df[str_cols].apply(lambda x: x.str.strip())

In [5]:
# Clean & recode Reference FCT column values
clean_dict = {'USDA': 'USDA_FCT', 'WAFCT': 'WA_FCT', 'West African FCT': 'WA_FCT', 'West African Table': 'WA_FCT',\
             'KFCT': 'Kenya_FCT', 'From WestAfrican FCT': 'WA_FCT', 'From the West Africa FCT': 'WA_FCT',\
             'WFCT': 'WA_FCT', 'From West African FCT': 'WA_FCT', 'Kenya NCT': 'Kenya_FCT', 'West African': 'WA_FCT',\
             'USDA FCT': 'USDA_FCT', 'FAFM': 'FAFH'}
matching_df.loc[:, 'Reference_Food_Composition_Table_FCT'] = matching_df.replace(clean_dict)

In [6]:
# Remove total row duplicates
matching_df = matching_df.drop_duplicates()

# Identify duplicate Food item code & desc
matching_df['duplicate_code'] = matching_df.Food_item_code_in_Household_Survey_item_cod.duplicated(keep='first')
matching_df['duplicate_desc'] = matching_df.Food_item_description_in_Household_Survey_desc.duplicated(keep='first')

# Get row numbers
print(matching_df[matching_df.duplicate_code == True].index)
print(matching_df[matching_df.duplicate_desc == True].index)

# Drop Food item code duplicate
matching_df = matching_df[matching_df.duplicate_code == False]

Int64Index([], dtype='int64')
Int64Index([196], dtype='int64')


In [7]:
matching_df = matching_df.drop(columns=['duplicate_code', 'duplicate_desc'])

In [8]:
# Quality Assurance Checks

# Check if any lines are duplicates
print('The are no duplicate rows: ' + str(sum(matching_df.duplicated()) == 0))

# Check that 'Food item code in Household Survey (item_cod)' is a unique identifier
print('Food_item_code_in_Household_Survey_item_cod is unique: {}'.format(matching_df['Food_item_code_in_Household_Survey_item_cod'].nunique() == matching_df.shape[0]))


The are no duplicate rows: True
Food_item_code_in_Household_Survey_item_cod is unique: True


### Read, clean and tidy West Africa FCT

In [9]:
# Read in WA FCT
wa_fct_df = pd.read_csv('./data/fct/WestAfricanFCT_userdatabase_cleaned.csv', header=0, encoding='utf-8')
wa_fct_df.shape

(470, 33)

In [10]:
# Clean Code column
wa_fct_df.loc[:, 'Code'] = wa_fct_df['Code'].str.strip()

In [11]:
# Drop Code duplicates
wa_fct_df['duplicate_code'] = wa_fct_df.Code.duplicated(keep='first')
# Drop Food item code duplicate
wa_fct_df = wa_fct_df[wa_fct_df.duplicate_code == False]
wa_fct_df.shape

(468, 34)

In [12]:
# Quality Assurance Checks
# Check if any lines are duplicates
print('The are no duplicate rows: ' + str(sum(wa_fct_df.duplicated()) == 0))
# Check that 'Code' is a unique identifier
print('Code is unique: {}'.format(wa_fct_df['Code'].nunique() == wa_fct_df.shape[0]))


The are no duplicate rows: True
Code is unique: True


In [13]:
# Drop extraneous cols
drop_cols = wa_fct_df.columns.tolist()[2:13] + ['duplicate_code']
wa_fct_df = wa_fct_df.drop(columns=drop_cols)
# Rename betacarotene
wa_fct_df = wa_fct_df.rename(columns=lambda x: re.sub('ß','beta',x))
wa_fct_df.rename(columns = {'Food name in English':'Food_description_in_FCT'}, inplace = True)

In [14]:
# Rename columns to match ADePT
# Read in cleaned col_names
wa_fct_adept_colnames = pd.read_csv('./variable_matching/wa_fct_adept_mn_col_names.csv')

# Create dict matching old with new
col_names_dict = dict(zip(wa_fct_adept_colnames.col_names, wa_fct_adept_colnames.adept_col_names))

# Rename with colnames
wa_fct_df = wa_fct_df.rename(columns = col_names_dict)

In [15]:
# Create wa copy
matching_wa_df = matching_df[matching_df.Reference_Food_Composition_Table_FCT == 'WA_FCT']

# Join WA FCT data to overall dataset to create WA only
wa_fct_dataset = pd.merge(matching_wa_df, wa_fct_df, left_on='Food_code_in_FCT', right_on='Code', how='left')
# Drop Code
wa_fct_dataset.drop(columns=['Code'], inplace=True)
wa_fct_dataset.shape

(101, 29)

### Read, clean and tidy Kenya FCT 

In [16]:
# Read in Kenya FCT
kenya_fct_df = pd.read_csv('./data/fct/Kenya Food Composition Tables 2018 Excel files Final-Revised mjm_2020-08-28.csv', header=0)

In [17]:
# Clean Code column
kenya_fct_df.loc[:, 'Code'] = kenya_fct_df['Code'].str.strip()

# Drop n and SD or min-max rows
print(kenya_fct_df.shape)
drop_n_min_max = ['n', 'SD or min-max']
kenya_fct_df = kenya_fct_df[~kenya_fct_df.Code.isin(drop_n_min_max)]
print(kenya_fct_df.shape)

(1222, 30)
(659, 30)


In [18]:
# Quality Assurance Checks

# Check if any lines are duplicates
print('The are no duplicate rows: ' + str(sum(kenya_fct_df.duplicated()) == 0))
# Check that 'Code' is a unique identifier
print('Code is unique: {}'.format(kenya_fct_df['Code'].nunique() == kenya_fct_df.shape[0]))

The are no duplicate rows: True
Code is unique: True


In [19]:
# Drop extraneous cols
drop_cols1 = list(kenya_fct_df)[2:11]
drop_cols2 = ['Food folate (mcg)', 'Vit A-RE (mcg)']
drop_cols = drop_cols1 + drop_cols2
kenya_fct_df = kenya_fct_df.drop(columns=drop_cols)
kenya_fct_df.rename(columns = {'Food name':'Food_description_in_FCT'}, inplace = True)

In [20]:
# Rename columns to match ADePT
# Read in cleaned col_names
kenya_fct_adept_colnames = pd.read_csv('./variable_matching/kenya_fct_adept_mn_col_names.csv')

# Create dict matching old with new
col_names_dict = dict(zip(kenya_fct_adept_colnames.col_names, kenya_fct_adept_colnames.adept_col_names))

# Rename with colnames
kenya_fct_df = kenya_fct_df.rename(columns = col_names_dict)

In [21]:
# Create kenya copy
matching_kenya_df = matching_df[matching_df.Reference_Food_Composition_Table_FCT == 'Kenya_FCT']

# Join Kenya FCT data to overall dataset to create Kenya only
kenya_fct_dataset = pd.merge(matching_kenya_df, kenya_fct_df, left_on='Food_code_in_FCT', right_on='Code', how='left')
# Drop Code
kenya_fct_dataset.drop(columns=['Code'], inplace=True)
kenya_fct_dataset.shape

(14, 26)

### Read, clean and tidy USDA FCT 

In [22]:
# Read in USDA FCT
usda_fct_df = pd.read_csv('./data/fct/USDA_SR24_ABBREV.csv', header=0)
usda_fct_df.shape

(7906, 53)

In [23]:
# Clean Code column
usda_fct_df.loc[:, 'NDB_No'] = usda_fct_df['NDB_No'].str.strip()

# Drop duplicate rows
usda_fct_df['duplicate'] = usda_fct_df.NDB_No.duplicated(keep='first')
usda_fct_df = usda_fct_df[usda_fct_df.duplicate == False]

In [24]:
# Quality Assurance Checks

# Check if any lines are duplicates
print('The are no duplicate rows: ' + str(sum(usda_fct_df.duplicated()) == 0))
# Check that 'Code' is a unique identifier
print('NDB_No is unique: {}'.format(usda_fct_df['NDB_No'].nunique() == usda_fct_df.shape[0]))

The are no duplicate rows: True
NDB_No is unique: True


In [25]:
# Rename æg
usda_fct_df = usda_fct_df.rename(columns=lambda x: re.sub('æg','mcg',x))

In [26]:
# Drop extraneous cols
drop_cols1 = list(usda_fct_df)[2:10] + ['duplicate']
drop_cols2 = list(usda_fct_df)[37:41]
drop_cols3 = ['Selenium_(mcg)', 'Panto_Acid_mg)', 'Folic_Acid_(mcg)', 'Food_Folate_(mcg)', 'Folate_DFE_(mcg)',\
              'Choline_Tot_ (mg)', 'Vit_A_IU', 'Alpha_Carot_(mcg)', 'Manganese_(mg)']
drop_cols4 = list(usda_fct_df)[42:]
drop_cols = drop_cols1 + drop_cols2 + drop_cols3 + drop_cols4
usda_fct_df = usda_fct_df.drop(columns=drop_cols)
usda_fct_df.rename(columns = {'Shrt_Desc':'Food_description_in_FCT'}, inplace = True)

In [27]:
# Rename columns to match ADePT
# Read in cleaned col_names
usda_fct_adept_colnames = pd.read_csv('./variable_matching/usda_fct_adept_mn_col_names.csv')

# Create dict matching old with new
col_names_dict = dict(zip(usda_fct_adept_colnames.col_names, usda_fct_adept_colnames.adept_col_names))

# Rename with colnames
usda_fct_df = usda_fct_df.rename(columns = col_names_dict)

In [28]:
# Create usda copy
matching_usda_df = matching_df[matching_df.Reference_Food_Composition_Table_FCT == 'USDA_FCT']

# Join USDA FCT data to overall dataset to create USDA only
usda_fct_dataset = pd.merge(matching_usda_df, usda_fct_df, left_on='Food_code_in_FCT', right_on='Code', how='left')
# Drop Code
usda_fct_dataset.drop(columns=['Code'], inplace=True)
usda_fct_dataset.shape

(72, 28)

### Read, clean and tidy SA FCT

In [29]:
# Read in SA FCT
sa_fct_df = pd.read_csv('./data/fct/sa_fct.csv', header=0, dtype={'Code': str})

In [30]:
# Clean Code column
sa_fct_df.loc[:,'Code'] = sa_fct_df['Code'].str.strip()

In [31]:
# Quality Assurance Checks

# Check if any lines are duplicates
print('The are no duplicate rows: ' + str(sum(sa_fct_df.duplicated()) == 0))
# Check that 'Code' is a unique identifier
print('Code is unique: {}'.format(sa_fct_df['Code'].nunique() == sa_fct_df.shape[0]))

The are no duplicate rows: True
Code is unique: True


In [32]:
# Drop extraneous cols
drop_cols1 = list(sa_fct_df)[2:17]
drop_cols2 = ['Mn-micro-g', 'Pant-mg', 'Biot-micro-g', 'E-mg']
drop_cols = drop_cols1 + drop_cols2
sa_fct_df = sa_fct_df.drop(columns=drop_cols)
sa_fct_df.rename(columns = {'Food Name':'Food_description_in_FCT'}, inplace = True)

In [33]:
# Read in cleaned col_names
sa_fct_adept_colnames = pd.read_csv('./variable_matching/sa_fct_adept_mn_col_names.csv')

# Create dict matching old with new
col_names_dict = dict(zip(sa_fct_adept_colnames['col_names'], sa_fct_adept_colnames['adept_col_names']))

# Rename with colnames
sa_fct_df = sa_fct_df.rename(columns = col_names_dict)

In [34]:
# Create SA copy
matching_sa_df = matching_df[matching_df.Reference_Food_Composition_Table_FCT == 'SA_FCT']

# Join SA FCT data to overall dataset to create SA only
sa_fct_dataset = pd.merge(matching_sa_df, sa_fct_df, left_on='Food_code_in_FCT', right_on='Code', how='left')
# Drop Code
sa_fct_dataset.drop(columns=['Code'], inplace=True)
sa_fct_dataset.shape

(12, 26)

## Combine four datasets into one 

In [35]:
# Create FAFH df
matching_fafh_df = matching_df[matching_df.Reference_Food_Composition_Table_FCT == 'FAFH']

In [36]:
total_fct_dataset = pd.concat([wa_fct_dataset, kenya_fct_dataset, usda_fct_dataset, sa_fct_dataset, matching_fafh_df], axis=0)

In [37]:
total_fct_dataset.to_csv('./output/total_fct_dataset.csv', index=False)

In [38]:
len(total_fct_dataset.columns.tolist())

30

In [39]:
# Reorder columns
adept_col_order_df = pd.read_csv('adept_col_order.csv', header=None)

adept_col_order_df[0].str.strip()

adept_col_order = adept_col_order_df[0].tolist()

In [40]:
# Reorder
total_fct_dataset = total_fct_dataset[adept_col_order]

In [41]:
# data_cols as string
data_cols = total_fct_dataset.columns.tolist()[9:]
total_fct_dataset[data_cols] = total_fct_dataset[data_cols].astype(str, errors='raise')

In [42]:
# Clean data_cols values
total_fct_dataset[data_cols] = total_fct_dataset[data_cols].apply(lambda x: x.str.replace('[', '').str.replace(']', '') if x.dtype == 'object' else x)

total_fct_dataset[data_cols] = total_fct_dataset[data_cols].replace({'nan':np.nan, 'Tr': 0, 'tr':0, '-':0, 'rt':0})

In [43]:
total_fct_dataset[data_cols].apply(lambda x: x.isna().sum())

Calcium (milligrams) (calcium)                      10
Iron (milligrams) (iron)                            10
Zinc (milligrams) (zinc)                            13
Folate (micrograms) (folate)                        23
Vitamin C (Ascorbic Acid) (milligrams) (vit_c)      14
Vitamin B1 (Thiamine) (milligrams) (vit_b1)         11
Vitamin B2 (Riboflavin) (milligrams) (vit_b2)       11
Retinol (micrograms) (retinol)                      27
Betacarotene equivalents (micrograms) (betacar)     41
Vitamin A (micrograms of RAE) (vita_rae)            17
Vitamin B6 (Pyridoxine) (milligrams) (vit_b6)       29
Vitamin B12 (Cobalamine) (micrograms) (vit_b12)     13
Magnesium (milligrams)                              10
Phosphorus (milligrams)                             10
Potassium (milligrams)                              13
Sodium (milligrams)                                 15
Copper (milligrams)                                 24
Vitamin D (micrograms)                              39
Vitamin E 

In [44]:
# Replace missing with 0s
total_fct_dataset = total_fct_dataset.drop(columns=['Selenium (micrograms)'])

In [45]:
# Convert data_cols to numeric
total_fct_dataset.sort_values(by=['Food_item_code_in_Household_Survey_item_cod']).to_csv('./output/Namibia_MN_fct_dataset_2020-10-31.csv', index=False, na_rep='')